In [1]:
%pip install numpy pandas matplotlib scikit-learn jinja2 seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42

In [3]:
df = pd.read_csv('.../DATA_Problematic Mobile Phone Use CAT.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '.../DATA_Problematic Mobile Phone Use CAT.csv'

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isna().sum())

df.describe().round(2)

In [ ]:
df['nmp_info'] = df[['NMP_1', 'NMP_2', 'NMP_3', 'NMP_4']].sum(axis=1)
df['nmp_conv'] = df[['NMP_5', 'NMP_6', 'NMP_7', 'NMP_8']].sum(axis=1)
df['nmp_cont'] = df[['NMP_9', 'NMP_10', 'NMP_11', 'NMP_12']].sum(axis=1)
df['nmp_conn'] = df[['NMP_13', 'NMP_14', 'NMP_15', 'NMP_16']].sum(axis=1)
df['nmp_total'] = df[['nmp_info', 'nmp_conv', 'nmp_cont', 'nmp_conn']].sum(axis=1)

df['nmp_norm'] = (
    (df['nmp_total'] - df['nmp_total'].min()) /
    (df['nmp_total'].max() - df['nmp_total'].min())
)

# SPAI subscales
df['spai_comp'] = df[['SPAI_3', 'SPAI_4', 'SPAI_5', 'SPAI_8', 'SPAI_9', 'SPAI_10', 'SPAI_15', 'SPAI_17']].sum(axis=1)
df['spai_imp'] = df[['SPAI_6', 'SPAI_10', 'SPAI_11', 'SPAI_12', 'SPAI_14', 'SPAI_18', 'SPAI_19']].sum(axis=1)
df['spai_with'] = df[['SPAI_2', 'SPAI_13', 'SPAI_16', 'SPAI_20']].sum(axis=1)
df['spai_tol'] = df[['SPAI_1', 'SPAI_7']].sum(axis=1)

# SPAI total
df['spai_total'] = df[['spai_comp', 'spai_imp', 'spai_with', 'spai_tol']].sum(axis=1)

# Normalise SPAI total (0-1)
df['spai_norm'] = (
    (df['spai_total'] - df['spai_total'].min()) /
    (df['spai_total'].max() - df['spai_total'].min())
)

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
#df['nmp-total'].hist(bins=20)
#%pip install seaborn
print('Mean:', df['nmp_norm'].mean())
sns.histplot(data=df, x='nmp_norm', bins=11, kde=True)

plt.xlabel('NMP Standardised Score')
plt.ylabel('Frequency')
plt.title('Distribution of NMP Standardised Score')
plt.xticks([0,0.5,1],
           ['Abscence of NMP', 'Neutral', 'Severe NMP']
           )
plt.axvline(df['nmp_norm'].mean(), color='red', linestyle='--')
plt.show()

In [ ]:
grade_mapping = {
    11: 1, 12: 2, 13: 3, 14: 4, 15: 5, 16: 6, #Grades 7-12
    1: 7, 2: 8, 3: 9, 4: 10, #Fresh-Senior
    5: 11, 6: 12, 7: 13, #Masters 1,2,3
    8: 14 #Doctor
}



df['grade_ord'] = df['grade'].map(grade_mapping)
df = df[df['grade_ord'] > 3]


df['grade_ord'].hist(bins=11)

df['grade_ord'].value_counts().sort_index()

df['grade_binary'] = np.where(df['grade'] > 10, 1, 0)
df['grade_binary'].value_counts()

In [ ]:
grade_mapping2 = {
    4:1, 5:1, 6:1, #Grades 9-12
    7:2, 8:2, 9:2, 10:2, #Fresh-Senior
    11:3, 12:3, 13:3, #Masters 1,2,3
    14:4 #Doctor
}

df['grade_ord_binned'] = df['grade_ord'].map(grade_mapping2)

df['grade_ord_binned'].hist(bins=4)

df['grade_ord_binned'].value_counts().sort_index()

In [ ]:

sns.histplot(data=df, x='nmp_norm', bins=20, kde=True)

plt.xlabel('NMP')
plt.ylabel('Frequency')
plt.title('Distribution of NMP')

plt.show()

In [ ]:

plt.scatter(df['nmp_norm'], df['grade_ord_binned'],
             alpha=0.3
            )

# Label the y-axis
plt.ylabel('grade (binned)')

# Set 4 y-axis ticks and their labels
plt.yticks(
    [1, 2, 3, 4],
    ['Secondary', 'College', 'Masters', 'Doctorate']
)

# Label the x-axis
plt.xlabel('NMP score')

# Set 3 x-axis ticks and their labels
plt.xticks(
    [0,0.5,1],
    ['Absence of nomophobia', 'Neutral', 'Severe Nomophobia']
)

NameError: name 'plt' is not defined

In [ ]:
X = df[["nmp_norm", "spai_norm"]]
X2 = df[[ "nmp_info", "nmp_conv", "nmp_cont", "nmp_conn"]]
y = df["grade_binary"]

class_names = {
    0: "2nd Level",
    1: "3rd Level"
}

X_train, X_test, X2_train, X2_test, y_train, y_test = train_test_split(
    X, X2, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)




print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

# Share of Pass/Fail in each group; stratify=y keeps these similar
pd.DataFrame({
    "All students": y.value_counts(normalize=True),
    "Training set": y_train.value_counts(normalize=True),
    "Test set": y_test.value_counts(normalize=True)
}).rename(index=class_names).round(2)

In [ ]:
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE)
)

log_reg.fit(X_train, y_train)

log_predictions = log_reg.predict(X_test)
log_probabilities = log_reg.predict_proba(X_test)[:, 1]

print("First 10 predicted classes:", log_predictions[:10])
print("First 10 probabilities of Pass:", np.round(log_probabilities[:10], 2))
print("Test accuracy:", round(accuracy_score(y_test, log_predictions), 3))

In [ ]:
log2_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE)
)

log2_reg.fit(X2_train, y_train)

log2_predictions = log2_reg.predict(X2_test)
log2_probabilities = log2_reg.predict_proba(X2_test)[:, 1]

print("First 10 predicted classes:", log2_predictions[:10])
print("First 10 probabilities of 3rd Lvl:", np.round(log2_probabilities[:10], 2))
print("Test accuracy:", round(accuracy_score(y_test, log2_predictions), 3))

In [ ]:
knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn.fit(X_train, y_train)
knn_predictions = knn.predict(X_test)

print("First 10 predicted classes:", knn_predictions[:10])
print("Test accuracy:", round(accuracy_score(y_test, knn_predictions), 3))

In [ ]:
knn2 = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn2.fit(X2_train, y_train)
knn2_predictions = knn2.predict(X2_test)

print("First 10 predicted classes:", knn2_predictions[:10])
print("Test accuracy:", round(accuracy_score(y_test, knn2_predictions), 3))

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN (k=5)", "KNN2", "LogReg2"],
    "Test accuracy": [
        accuracy_score(y_test, log_predictions),
        accuracy_score(y_test, knn_predictions),
        accuracy_score(y_test, knn2_predictions),
        accuracy_score(y_test, log2_predictions),
    ]
}).sort_values("Test accuracy", ascending=False)

results

In [ ]:
def plot_decision_boundary(model, X_train, y_train, title, ax):

    x_min, x_max = X_train["nmp_norm"].min() - 0.1, X_train["nmp_norm"].max() + 0.1

    y_min, y_max = X_train["spai_norm"].min() - 0.1, X_train["spai_norm"].max() + 0.1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250)
    )

    grid = pd.DataFrame({
        "nmp_norm": xx.ravel(),
        "spai_norm": yy.ravel()
    })

    zz = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.18, levels=[0, 0.1, 1])

    for label, name in [(0, "Fail"), (1, "Pass")]:
        subset = X_train.loc[y_train == label]
        ax.scatter(
            subset["nmp_norm"],
            subset["spai_norm"],
            label=name,
            alpha=0.8
        )

    ax.set_xlabel("Normalized NMP")
    ax.set_ylabel("Normalized SPAI")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

plot_decision_boundary(log_reg, X_train, y_train, "Logistic Regression", axes[0])
plot_decision_boundary(knn, X_train, y_train, "KNN (k=5)", axes[1])

axes[0].legend()
plt.tight_layout(pad=0)
plt.show()

In [ ]:
models = {
    "Logistic Regression": (log_reg, X_test),
    "KNN (NMP + SPAI)": (knn, X_test),
    "KNN (NMP Subscales)": (knn2, X2_test)
}

validation_rows = []

for name, (model, X_val) in models.items():
    pred = model.predict(X_val)

    validation_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

validation_results = (
    pd.DataFrame(validation_rows)
    .set_index("Model")
    .sort_values("F1", ascending=False)
)

validation_results.round(3)

In [ ]:
# Choose the model with the highest validation F1 score
best_name = validation_results.index[0]

best_model, best_X_test = models[best_name]

valid_pred = best_model.predict(best_X_test)

print("Selected using validation F1:", best_name)
print("Confusion matrix (actual rows, predicted columns):")

print(confusion_matrix(y_test, valid_pred, labels=[0, 1]))

ConfusionMatrixDisplay.from_predictions(
    y_test, valid_pred,
    labels=[0, 1],
    display_labels=["2nd Level (0)", "3rd Level (1)"],
    cmap="Blues"
)

plt.title(f"Test confusion matrix: {best_name}")
plt.show()